In [ ]:
%cd ..
import os, sys
# Must be set before pyproj is imported on Windows
os.environ["PROJ_DATA"] = os.path.join(sys.prefix, "Library", "share", "proj")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyproj import Transformer
from skgstat import Variogram
from statsmodels.tsa.stattools import acf

import config
from utils.data import load_dataset

%matplotlib inline
sns.set_theme(style="whitegrid")

In [ ]:
# Load data
df = load_dataset()

# Ensure output dirs exist
os.makedirs(config.output_dir_correlations, exist_ok=True)

# Variogram Analysis

In [ ]:
decorrelation_ranges = []

for param in config.parameters:
    print(param)
    # Subsample for computational tractability
    df_sub = df[["LONGITUDE", "LATITUDE", "LEV_M", "DATEANDTIME", param]].dropna().sample(n=10000, random_state=42)

    # Project lat/lon to metric distances (Lambert Azimuthal Equal Area, centred on the data)
    transformer = Transformer.from_crs(
        "+proj=longlat +datum=WGS84",
        f"+proj=laea +lat_0={df_sub['LATITUDE'].mean()} +lon_0={df_sub['LONGITUDE'].mean()} +datum=WGS84 +units=m +no_defs",
        always_xy=True,
    )
    x, y = transformer.transform(df_sub["LONGITUDE"], df_sub["LATITUDE"])
    df_sub["LONGITUDE"] = x / 1000  # metres → km
    df_sub["LATITUDE"] = y / 1000

    # Fit variograms along the three independent dimensions
    spatial_v = Variogram(coordinates=df_sub[["LONGITUDE", "LATITUDE"]].values, values=df_sub[param].values, use_nugget=True)
    depth_v   = Variogram(coordinates=df_sub[["LEV_M"]].values,      values=df_sub[param].values, use_nugget=True)
    time_v    = Variogram(coordinates=df_sub[["DATEANDTIME"]].values, values=df_sub[param].values, use_nugget=True)

    for v, xlabel, fname in [
        (spatial_v, "Distance [km]",    f"spatial_variogram_{param}.png"),
        (depth_v,   "Distance [m]",     f"depth_variogram_{param}.png"),
        (time_v,    "Distance [years]", f"time_variogram_{param}.png"),
    ]:
        fig = v.plot()
        fig.axes[1].set_ylabel("N samples")
        fig.axes[0].set_xlabel(xlabel)
        fig.axes[0].set_ylabel("Semivariance")
        plt.tight_layout()
        # plt.savefig(os.path.join(config.output_dir_correlations, fname), dpi=300)
        plt.show()

    spatial_range, spatial_sill, spatial_nugget = spatial_v.parameters
    depth_range,   depth_sill,   depth_nugget   = depth_v.parameters
    time_range,    time_sill,    time_nugget     = time_v.parameters

    decorrelation_ranges.append(pd.DataFrame({
        "parameter":      [param],
        "spatial_range":  [spatial_range],  "spatial_sill":  [spatial_sill],  "spatial_nugget":  [spatial_nugget],
        "depth_range":    [depth_range],    "depth_sill":    [depth_sill],    "depth_nugget":    [depth_nugget],
        "time_range":     [time_range],     "time_sill":     [time_sill],     "time_nugget":     [time_nugget],
    }))

df_decorrelation_ranges = pd.concat(decorrelation_ranges, ignore_index=True)
df_decorrelation_ranges.to_csv(os.path.join(config.output_dir_correlations, "variogram_data.csv"), index=False)

In [ ]:
df_decorrelation_ranges

## Results Table

In [ ]:
# Beautify naming
df_decorrelation_ranges["parameter"] = df_decorrelation_ranges["parameter"].map(config.parameter_name_map)
df_decorrelation_ranges = df_decorrelation_ranges.rename(columns={"spatial_range": "Spatial range [km]", "spatial_sill": "Spatial sill", "spatial_nugget": "Spatial nugget",
       "depth_range": "Depth range [m]", "depth_sill": "Depth sill", "depth_nugget": "Depth nugget", "time_range": "Time range [years]", "time_sill": "Time sill",
       "time_nugget":"Time nugget"})

formatters = {
        "Spatial range [km]": "{:.0f}".format,
        "Spatial sill": "{:.1f}".format,
        "Spatial nugget": "{:.1f}".format,
        "Depth range [m]": "{:.0f}".format,
        "Depth sill": "{:.1f}".format,
        "Depth nugget": "{:.1f}".format,
        "Time range [years]": "{:.0f}".format,
        "Time sill": "{:.1f}".format,
        "Time nugget": "{:.1f}".format
    }

In [ ]:
df_formatted = df_decorrelation_ranges.copy()
for col, fmt in formatters.items():
    df_formatted[col] = df_formatted[col].map(fmt)

# Transpose so parameters are columns and statistics are rows
df_transposed = df_formatted.set_index("parameter").T
n_params = len(df_transposed.columns)
tex_table = df_transposed.to_latex(
    index=True,
    column_format="l|" + "r" * n_params,
    caption="Summary of variogram statistics per parameter",
    label="tab:variogram_results",
)

# Replace the rule-separated first header cell with \multicolumn to avoid a double vertical line
lines = tex_table.split("\n")
header_idx = next(i + 1 for i, l in enumerate(lines) if "\\toprule" in l)
header = lines[header_idx]
sep = header.index("&")
first_cell = header[:sep].strip()
lines[header_idx] = f"\\multicolumn{{1}}{{l}}{{{first_cell}}} &" + header[sep + 1:]
print("\n".join(lines))

# Temporal Autocorrelation

In [ ]:
temporal_autocorrelations = []

for param in config.parameters:
    # Average over all locations per time step before computing ACF to isolate the temporal signal
    time_means = df[["DATEANDTIME", param]].dropna().groupby("DATEANDTIME")[param].mean()
    acf_vals = acf(time_means.values, fft=True)
    temporal_autocorrelations.append(
        pd.DataFrame({"parameter": [param] * len(acf_vals), "acf": acf_vals, "lag": range(len(acf_vals))})
    )

df_autocorr = pd.concat(temporal_autocorrelations, ignore_index=True)
df_autocorr.to_csv(os.path.join(config.output_dir_correlations, "temporal_autocorrelations.csv"), index=False)

pivot = df_autocorr.copy()
pivot["parameter"] = pivot["parameter"].map(config.parameter_name_map)
pivot = pivot.groupby(["parameter", "lag"])["acf"].mean().unstack()

# Final printed size (supplement text width = 6.69 in)
fontsize = 8
figsize = (6.69, 3.0)

# Plot
plt.figure(figsize=figsize)
ax = sns.heatmap(pivot, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": fontsize})
ax.collections[0].colorbar.ax.tick_params(labelsize=fontsize)
ax.tick_params(labelsize=fontsize)
plt.xlabel("Lag [20-year time steps]", fontsize=fontsize)
plt.ylabel("")
plt.grid(False)
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir_correlations, "temporal_autocorrelation.pdf"), dpi=1000)
plt.show()